In [1]:
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
from StocProcess.RBM import MakeRBMTransProbFunc
from QAE.LowDepthQAE import LowDepthQAE

In [2]:
# RBM parameters
c = -1
d = 1
x0 = 0.5 * (c + d)
t0 = 0
t = 0.6
mu = 0.5
sigma = 1.0
n_terms = 100

# QAE setting
nShot = 12
epsilon = 2**(-5)
nRep = 100

In [3]:
# PDF at time t
transProbFunc = MakeRBMTransProbFunc(t, t0, c, d, mu, sigma, n_terms)

# Prob(X > (c + d)/2)
integFunc = lambda x: transProbFunc(x, x0)
pTrue, _ = sp.integrate.quad(integFunc, x0, 1.0)

In [4]:
Ns = (2 ** np.linspace(3, 7, 9)).astype(int)
print(Ns)

[  8  11  16  22  32  45  64  90 128]


In [6]:
retDf = pd.DataFrame(columns=['N', 'pTrue', 'beta', 'pEst', 'absErr', 'totalQueryNum', 'maxDepth'])

for _ in range(nRep):
    for iN in range(len(Ns)):
        N = Ns[iN]
        beta = np.log(N**0.5) / np.log(1 / epsilon)
        qaeRes = LowDepthQAE(pTrue, epsilon, nShot, beta)
        pEst = qaeRes.aEst
        totalQueryNum = qaeRes.TotalQueryNum * N * (N+1) / 2
        maxDepth = qaeRes.MaxDepth * N
        retDf.loc[len(retDf)] = [N, pTrue, beta, pEst, abs(pEst - pTrue), totalQueryNum, maxDepth]

In [7]:
retDf

,N,pTrue,beta,pEst,absErr,totalQueryNum,maxDepth
0,8.0,0.649605,0.300000,0.652665,0.003060,53136.0,200.0
1,11.0,0.649605,0.345943,0.648965,0.000640,109296.0,231.0
2,16.0,0.649605,0.400000,0.649265,0.000340,254592.0,272.0
3,22.0,0.649605,0.445943,0.655066,0.005461,582912.0,286.0
4,32.0,0.649605,0.500000,0.648965,0.000640,1596672.0,352.0
...,...,...,...,...,...,...,...
895,32.0,0.649605,0.500000,0.653965,0.004361,1596672.0,352.0
896,45.0,0.649605,0.549185,0.650765,0.001160,3949560.0,405.0
897,64.0,0.649605,0.600000,0.659666,0.010061,9584640.0,576.0
898,90.0,0.649605,0.649185,0.648565,0.001040,24324300.0,630.0


In [ ]:
retDf.groupby('N')['absErr'].mean()

In [10]:
retDf.to_csv('RBM_LowDepthQAE.csv', index=False)